# Saving a query as a view

Practice notebook for the datafromzero SQL lesson **Saving a query as a view**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

Views live in the notebook's in-memory database. If the notebook restarts, run the setup cell and the `CREATE` cells again.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Write the query once

In [ ]:
con.sql("""
SELECT i.order_item_id, o.order_id, CAST(o.ordered_at AS DATE) AS order_day, o.store_id,
       p.product_name, p.category, i.quantity,
       i.quantity * p.unit_price_usd - i.discount_usd AS amount_usd
FROM order_items AS i
JOIN orders AS o ON o.order_id = i.order_id
JOIN products AS p ON p.product_id = i.product_id
WHERE o.status = 'completed'
ORDER BY i.order_item_id
LIMIT 5
""").show()

### Step 2. Save it as a view

The same query without `ORDER BY` and `LIMIT`, after `CREATE OR REPLACE VIEW`. The empty table with a `Count` column is the expected result.

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW sales_lines AS
SELECT i.order_item_id, o.order_id, CAST(o.ordered_at AS DATE) AS order_day, o.store_id,
       p.product_name, p.category, i.quantity,
       i.quantity * p.unit_price_usd - i.discount_usd AS amount_usd
FROM order_items AS i
JOIN orders AS o ON o.order_id = i.order_id
JOIN products AS p ON p.product_id = i.product_id
WHERE o.status = 'completed'
""").show()

### Step 3. Read the view like a table

In [ ]:
con.sql("""
SELECT store_id, COUNT(*) AS lines, ROUND(SUM(amount_usd), 2) AS sales
FROM sales_lines
GROUP BY store_id
ORDER BY store_id
""").show()

In [ ]:
con.sql("""
SELECT category, COUNT(*) AS lines, ROUND(SUM(amount_usd), 2) AS sales
FROM sales_lines
GROUP BY category
ORDER BY category
""").show()

### Step 4. Verify

In [ ]:
con.sql("""
SELECT COUNT(*) AS lines, ROUND(SUM(amount_usd), 2) AS sales FROM sales_lines
""").show()

In [ ]:
con.sql("""
SELECT ROUND(SUM(i.quantity * p.unit_price_usd - i.discount_usd), 2) AS all_sales
FROM order_items AS i
JOIN products AS p ON p.product_id = i.product_id
""").show()

### Step 5. See what is stored

In [ ]:
con.sql("""
SELECT sql FROM duckdb_views() WHERE view_name = 'sales_lines'
""").show()

### Step 6. Share only some columns

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW orders_shared AS
SELECT order_id, store_id, ordered_at, status, payment_method FROM orders
""").show()

In [ ]:
con.sql("""
SELECT * FROM orders_shared LIMIT 3
""").show()

### Step 7. Views live in the database that made them

A second connection is a second, empty in-memory database. Expect a `Catalog Error`. The notebook prints the error instead of stopping.

In [ ]:
other = duckdb.connect()   # a second, empty in-memory database
try:
    other.sql("""
SELECT COUNT(*) FROM sales_lines
""").show()
except Exception as e:
    print(e)

### Step 8. A trigger: know the word

Expect a Parser Error: DuckDB 2.0.0 does not accept this statement. The notebook prints the error instead of stopping.

In [ ]:
try:
    con.sql("""
CREATE TRIGGER log_order AFTER INSERT ON orders
BEGIN
  SELECT 1;
END
""").show()
except Exception as e:
    print(e)

## Exercise 1. Daily sales as a view on a view

Needs the view `sales_lines` from step 2: run that cell first. Finish the view: add the number of lines and the rounded sum of `amount_usd`, named `lines` and `sales`. Then finish the check query.

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW daily_sales AS
SELECT order_day
       -- add the two aggregates here, separated by commas
FROM sales_lines
GROUP BY order_day
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS days
       -- add the sum of sales and the sum of lines
FROM daily_sales
""").show()

## Exercise 2. How many completed orders did each store take, using `sales_lines`?

Before you run anything, write one sentence in the cell below: what one row of `sales_lines` is, and what that means for counting orders. Then write the query, and check it against a count straight from `orders`.

In [ ]:
# My sentence:
#

In [ ]:
# Query using the view:

In [ ]:
# Check against orders:

## Exercise 3. Spot the flaw

The team reads a view `team_sales`. The first cell builds it for you. A colleague wants the canceled lines too and replaces it (second cell). Run the check (third cell): what went wrong, and what should they have done?

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW team_sales AS
SELECT i.order_item_id, o.order_id, CAST(o.ordered_at AS DATE) AS order_day, o.store_id,
       p.product_name, p.category, i.quantity,
       i.quantity * p.unit_price_usd - i.discount_usd AS amount_usd
FROM order_items AS i
JOIN orders AS o ON o.order_id = i.order_id
JOIN products AS p ON p.product_id = i.product_id
WHERE o.status = 'completed'
""").show()

In [ ]:
con.sql("""
SELECT ROUND(SUM(amount_usd), 2) AS sales FROM team_sales
""").show()

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW team_sales AS
SELECT i.order_item_id, o.order_id, CAST(o.ordered_at AS DATE) AS order_day, o.store_id,
       p.product_name, p.category, i.quantity,
       i.quantity * p.unit_price_usd - i.discount_usd AS amount_usd
FROM order_items AS i
JOIN orders AS o ON o.order_id = i.order_id
JOIN products AS p ON p.product_id = i.product_id
""").show()

In [ ]:
con.sql("""
SELECT ROUND(SUM(amount_usd), 2) AS sales FROM team_sales
""").show()

In [ ]:
# Your fix: